In [4]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


def train_ticket_classifier(csv_path, model_path="outputs/ticket_model.joblib"):
    df = pd.read_csv(csv_path)

    X = df["ticket_text"]
    y = df["category"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2)
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    print(classification_report(y_test, predictions))

    joblib.dump(model, model_path)

    return model


def predict_ticket(model, ticket_text):
    prediction = model.predict([ticket_text])[0]
    probabilities = model.predict_proba([ticket_text])[0]

    confidence = probabilities.max()

    return {
        "ticket_text": ticket_text,
        "predicted_category": prediction,
        "confidence": round(confidence, 3)
    }


if __name__ == "__main__":
    model = train_ticket_classifier("data/student_tickets.csv")

    sample_ticket = "I cannot register for my course because the system shows an error"
    result = predict_ticket(model, sample_ticket)

    print(result)

                     precision    recall  f1-score   support

  academic_advising       0.33      0.50      0.40         2
         admissions       1.00      1.00      1.00         2
course_registration       0.00      0.00      0.00         3
      financial_aid       0.00      0.00      0.00         3
         graduation       0.75      1.00      0.86         3
  technical_support       0.50      1.00      0.67         3

           accuracy                           0.56        16
          macro avg       0.43      0.58      0.49        16
       weighted avg       0.40      0.56      0.46        16

{'ticket_text': 'I cannot register for my course because the system shows an error', 'predicted_category': 'course_registration', 'confidence': 0.235}


C:\Users\fedab\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\fedab\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\fedab\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
